# Step 1: Raw Data Exploration & Cropping Preview
This cell generates a quick, uncropped plot of the raw CSV data. The objective is to visually inspect the continuous waveform and identify the exact start and end times (in seconds) of the clean movement cycles you want to analyze. Take note of these timestamps to manually configure the parameters in the next step.

In [3]:
import pandas as pd
import plotly.express as px

file_path = 'S06_Shoulder_Abduction_60bpm.csv'

df = pd.read_csv(file_path)

# Extraemos todas las columnas excepto el tiempo
angle_columns = [col for col in df.columns if col != 'Time(s)']

# Pintamos todas las curvas para explorar el archivo
fig = px.line(df, x='Time(s)', y=angle_columns,
              title=f'Graph: {file_path}',
              labels={'Time(s)': 'Time (seconds)', 'value': 'Angle (degrees)', 'variable': 'Movement Axis'})

fig.update_traces(mode='lines+markers', marker=dict(size=4))
fig.show()

print("Observe the graph and decide the exact start and end seconds you want to crop.")

Observe the graph and decide the exact start and end seconds you want to crop.


### Step 2: Clinical Kinematic Analysis & Validation
This is the main analysis program. Before running this cell, update the **Analysis Params** variables in the code with the start and end times identified in Step 1, along with the specific clinical parameters for this trial (BPM, baseline resting angle, and target maximum angle).

Based on your inputs, the script will:
1. Apply a zero-phase low-pass Butterworth filter to remove IMU high-frequency noise.
2. Crop the dataset to your specified timeframe.
3. Generate the theoretical ideal wave based on the metronome pace.
4. Calculate key validation metrics (RMSE, Pearson *r* correlation, and Peak Error).
5. Output the final comparative graph.



In [1]:
# SET UP AND LIBRARIES
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.stats import pearsonr
import ipywidgets as widgets
from IPython.display import display

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded successfully. Ready for Data Analysis.")

Libraries loaded successfully. Ready for Data Analysis.


In [4]:
# ANALYSIS PARAMS - ADJUST FOR EACH CSV

csv_file_path = 'S06_Shoulder_Abduction_60bpm.csv'

#Select evaluation movement
      #Shoulder_Flexion_Deg
      #Shoulder_Abduction_Deg
      #Elbow_Flexion_Deg
      #Elbow_Pronosupination_Deg

target_axis = 'Shoulder_Abduction_Deg'

# Manual time crop
start_time = 3.235
end_time = 13.174

# Subject Clinical Parameters
metronome_bpm = 60
baseline_resting_angle = 0.2   # Real "0" (what the IMU reads at rest)
target_max_angle = 90          # Target degree for that movement



In [5]:
#FUNCTIONS

#Applies a zero-phase low-pass Butterworth filter to remove high-frequency IMU noise.
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs # Nyquist Frequency
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return filtfilt(b, a, data)

# Generates the theoretical 'Ideal Curve' based on the metronome pace.
def generate_ideal_curve(time_array, target_min, target_max, bpm, time_offset):
    freq_hz = (bpm / 60) / 2  #1 full cycle (up and down) equals 2 beats
    amplitude = (target_max - target_min) / 2
    offset = target_min + amplitude
    # Generate a sine wave matching the metronome frequency
    t_shifted = time_array - time_offset
    ideal_wave = offset - amplitude * np.cos(2 * np.pi * freq_hz * t_shifted)
    return ideal_wave


In [7]:
#DATA PROCESSING

df_raw = pd.read_csv(csv_file_path)

# Extract available angle columns dynamically
available_movements = [col for col in df_raw.columns if col != 'Time(s)']


#Manual crop
df_cut = df_raw[(df_raw['Time(s)'] >= start_time) & (df_raw['Time(s)'] <= end_time)].copy()

if df_cut.empty:
    print("Error! The crop times are out of the file's bounds.")
else:
    # Reset the timeline so the cropped section starts at 0s
    df_cut['Time_Reset'] = df_cut['Time(s)'] - df_cut['Time(s)'].iloc[0]
    time_array = df_cut['Time_Reset'].values

    # Estimate sampling rate (FPS)
    time_diffs = np.diff(df_cut['Time(s)'])
    fs_estimated = 1.0 / np.mean(time_diffs)


    # Noise filtering
    unity_curve = butter_lowpass_filter(df_cut[target_axis], cutoff=5.0, fs=fs_estimated)
    raw_curve = df_cut[target_axis].values


In [8]:
#DATA ANALYSIS

def interactive_tuner(bpm_adj, min_adj, max_adj, time_offset):

    # Generate ideal curve
    ideal_curve_adj = generate_ideal_curve(time_array, min_adj, max_adj, bpm_adj, time_offset)

    # Recalculate metrics
    rmse_adj = np.sqrt(np.mean((unity_curve - ideal_curve_adj)**2))
    peak_error_adj = np.max(np.abs(unity_curve - ideal_curve_adj))
    pearson_r_adj, _ = pearsonr(unity_curve, ideal_curve_adj)

    # Plot the curves
    plt.figure(figsize=(12, 6))
    plt.plot(time_array, raw_curve, color='lightgray', label=f'Raw Data ({target_axis})', alpha=0.6)
    plt.plot(time_array, unity_curve, color='#1f77b4', label=f'Filtered Signal', linewidth=2.5)
    plt.plot(time_array, ideal_curve_adj, color='#d62728', linestyle='--', label=f'Ideal Target ({bpm_adj:.1f} BPM)', linewidth=2)

    plt.title(f'{csv_file_path.replace(".csv", "")} - {target_axis}', pad=15, fontweight='bold')
    plt.xlabel('Time (seconds)', fontweight='bold')
    plt.ylabel('Angle (degrees)', fontweight='bold')
    plt.legend(loc='upper right', frameon=True, shadow=True)
    plt.grid(True, linestyle=':', alpha=0.7)

    # Show metrics in image
    metrics_text = (
            f"RMSE: {rmse_adj:.2f}º   |   "
            f"Peak Error: {peak_error_adj:.2f}º   |   "
            f"Pearson r: {pearson_r_adj:.4f}")

    plt.subplots_adjust(bottom=0.2)
    metrics_box = plt.gca().text(0.5, -0.15, metrics_text, transform=plt.gca().transAxes,
                      fontsize=11, ha='center', va='top',
                      bbox=dict(boxstyle='round,pad=0.6', facecolor='#f8f9fa', edgecolor='gray'))
    margin = plt.gca().text(0.5, -0.24, " ", transform=plt.gca().transAxes)

    plt.show()

print("Slide to adjust parameters")

_ = widgets.interact(interactive_tuner,
                 bpm_adj=widgets.FloatSlider(value=metronome_bpm, min=30, max=140, step=0.5, description='BPM:', layout={'width': '500px'}),
                 min_adj=widgets.FloatSlider(value=baseline_resting_angle, min=-100, max=90, step=0.5, description='Min Angle:', layout={'width': '500px'}),
                 max_adj=widgets.FloatSlider(value=target_max_angle, min=0, max=180, step=0.5, description='Max Angle:', layout={'width': '500px'}),
                 time_offset=widgets.FloatSlider(value=0.0, min=-1.5, max=1.5, step=0.02, description='Offset (sec):', layout={'width': '500px'}))

Slide to adjust parameters


interactive(children=(FloatSlider(value=60.0, description='BPM:', layout=Layout(width='500px'), max=140.0, min…